# Ad Click Prediction - CTR Modeling Submission

**Goal:** Predict ad clicks and convert model insights into targeting, bidding, campaign, and inventory actions.

**Training data:** 463,291 impressions, 31,331 clicks, **CTR 6.76%**.  
**Test data:** 128,858 unlabeled impressions.  
**Validation design:** Train on July 2-5, tune/threshold on July 6, untouched temporal holdout on July 7. The provided test data begins July 8.

> **Final recommendation:** CatBoost with personalization and interaction features. It delivers the best temporal holdout ROC-AUC (**0.580**) and PR-AUC (**0.078**) while avoiding the training-data expansion of SMOTE.

## 1. Data understanding and preparation

In [ ]:
import pandas as pd, numpy as np
train = pd.read_csv("Ad_click_prediction_train (1).csv", parse_dates=["DateTime"])
test = pd.read_csv("Ad_Click_prediciton_test.csv", parse_dates=["DateTime"])

# Inspect structure, target imbalance, missingness and date range
print(train.shape, test.shape)
print(train["is_click"].value_counts(normalize=True))
print(train.isna().sum().sort_values(ascending=False))

**Observed data quality**

- `product_category_2`: 365,854 missing (79.0%).
- `city_development_index`: 125,129 missing (27.0%).
- `user_group_id`, `gender`, `age_level`, `user_depth`: 18,243 missing each (3.9%).

Identifiers and coded demographic fields are treated as **categorical variables**, so missing values are represented as an explicit `MISSING` level rather than median-imputing a fake category.

![EDA summary](ctr_eda.png)

## 2. Feature engineering and leakage control

In [ ]:
def add_features(df):
    x = df.copy()
    x["hour"] = x.DateTime.dt.hour
    x["minute"] = x.DateTime.dt.minute
    x["dow"] = x.DateTime.dt.dayofweek
    x["is_weekend"] = (x["dow"] >= 5).astype(int)
    x["daypart"] = pd.cut(x.hour, [-1,5,11,16,20,23],
                          labels=["night","morning","afternoon","evening","late_evening"]).astype(str)
    x["user_product"] = x.user_id.astype(str) + "|" + x["product"].astype(str)
    x["campaign_webpage"] = x.campaign_id.astype(str) + "|" + x.webpage_id.astype(str)
    x["gender_age"] = x.gender.fillna("MISSING").astype(str) + "|" + x.age_level.fillna(-1).astype(str)
    return x

# IMPORTANT: target aggregates (user_ctr/product_ctr) must be computed only from past data.
# Never calculate them on the full labeled dataset before splitting.
model_df = add_features(train)

**Why this matters:** `user_ctr`, `product_ctr`, campaign CTR, etc. are powerful, but calculating them from all labeled rows before validation leaks future outcomes. For production, use past-only feature-store aggregates, out-of-fold encodings, or CatBoost ordered target statistics.

## 3. Business EDA

### Weekend vs weekday

|    | is_weekend   |   impressions |   clicks | ctr   |
|---:|:-------------|--------------:|---------:|:------|
|  0 | False        |        384246 |    25540 | 6.65% |
|  1 | True         |         79045 |     5791 | 7.33% |

Weekend CTR is **7.33%** vs weekday **6.65%** - about **10.2% relative uplift**. Chi-square p-value = **4.51e-12**.

**Caution:** the labeled data contains only one weekend day (Sunday), so this is directional rather than a multi-week causal conclusion.

### Product performance

|    | product   |   impressions |   clicks | ctr   | click_share   |   ctr_index |
|---:|:----------|--------------:|---------:|:------|:--------------|------------:|
|  0 | J         |          9698 |      899 | 9.27% | 2.87%         |    137.075  |
|  1 | D         |         41064 |     2949 | 7.18% | 9.41%         |    106.192  |
|  2 | H         |        109574 |     7654 | 6.99% | 24.43%        |    103.291  |
|  3 | C         |        163501 |    11306 | 6.91% | 36.09%        |    102.251  |
|  4 | E         |         21452 |     1474 | 6.87% | 4.70%         |    101.604  |
|  5 | I         |         63711 |     4079 | 6.40% | 13.02%        |     94.6714 |
|  6 | A         |         15391 |      953 | 6.19% | 3.04%         |     91.56   |
|  7 | B         |         22479 |     1238 | 5.51% | 3.95%         |     81.4373 |
|  8 | F         |          7007 |      344 | 4.91% | 1.10%         |     72.5949 |
|  9 | G         |          9414 |      435 | 4.62% | 1.39%         |     68.3274 |

Product **J** has the highest CTR (9.27%), while **G** and **F** are weakest. Product **C** produces the most absolute clicks because of scale.

### High-propensity user profiles (minimum 1,000 impressions)

|    | gender   |   age_level |   city_development_index |   impressions |   clicks | ctr   |
|---:|:---------|------------:|-------------------------:|--------------:|---------:|:------|
| 63 | Male     |           5 |                        4 |          2945 |      255 | 8.66% |
| 28 | Female   |           5 |                        4 |          1181 |      100 | 8.47% |
| 29 | Female   |           5 |                      nan |          2717 |      220 | 8.10% |
| 44 | Male     |           1 |                      nan |         19921 |     1526 | 7.66% |
| 41 | Male     |           1 |                        2 |          7922 |      605 | 7.64% |
| 26 | Female   |           5 |                        2 |          2752 |      208 | 7.56% |
| 64 | Male     |           5 |                      nan |          5703 |      431 | 7.56% |
| 46 | Male     |           2 |                        2 |         50409 |     3736 | 7.41% |

## 4. Model design

In [ ]:
# Temporal split
model_df["split_date"] = model_df.DateTime.dt.strftime("%Y-%m-%d")
core = model_df[model_df.split_date <= "2017-07-05"]
cal  = model_df[model_df.split_date == "2017-07-06"]
val  = model_df[model_df.split_date == "2017-07-07"]

# Models compared:
# 1) Logistic Regression + one-hot + class_weight=balanced
# 2) LightGBM + categorical encoding + class_weight=balanced
# 3) CatBoost + user_id + user_product + campaign_webpage + gender_age
# Threshold is selected on Jul 6 by maximum F1; Jul 7 remains untouched.

### Reproducible Logistic Regression baseline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), ["hour","minute"])
])
logit = Pipeline([
    ("pre", pre),
    ("model", LogisticRegression(class_weight="balanced", solver="liblinear", C=.5, max_iter=100))
])
logit.fit(core[features], core.is_click)

### Reproducible LightGBM baseline

In [ ]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(
    n_estimators=600, learning_rate=.04, num_leaves=31,
    min_child_samples=100, class_weight="balanced", random_state=42, n_jobs=-1
)
lgb.fit(X_core, core.is_click)

### Reproducible CatBoost model

In [ ]:
from catboost import CatBoostClassifier
cat_features = [
    "product","campaign_id","webpage_id","product_category_1","product_category_2",
    "user_group_id","gender","age_level","user_depth","city_development_index","var_1",
    "dow","is_weekend","daypart","user_id","user_product","campaign_webpage","gender_age"
]
model = CatBoostClassifier(
    iterations=49, depth=6, learning_rate=.08, l2_leaf_reg=5, random_strength=1,
    loss_function="Logloss", eval_metric="AUC", random_seed=42, verbose=False
)
model.fit(X_core, core.is_click, cat_features=cat_features)

## 5. Model evaluation

|    | model                      |   ROC-AUC |   PR-AUC |   Precision |   Recall |      F1 |
|---:|:---------------------------|----------:|---------:|------------:|---------:|--------:|
|  0 | Logistic Regression        |  0.525902 | 0.065134 |     0.06605 |  0.64463 | 0.11982 |
|  1 | LightGBM                   |  0.543092 | 0.068699 |     0.06842 |  0.60759 | 0.12299 |
|  2 | CatBoost + personalization |  0.579538 | 0.078022 |     0.0781  |  0.48534 | 0.13455 |

![Model comparison](ctr_models.png)

**Selected model:** CatBoost + personalization. Relative to the Logistic Regression baseline, ROC-AUC improves by about **10.2%** and PR-AUC by about **19.8%**. The absolute scores are not extremely high, which is itself useful evidence that short-window CTR prediction is noisy and should be monitored continuously.

## 6. Does personalization help?

Ablation test using the same CatBoost settings and July 2-6 training window:

|    | feature_set        |   ROC-AUC |   PR-AUC |
|---:|:-------------------|----------:|---------:|
|  0 | Base               |  0.550399 | 0.072366 |
|  1 | + user_id          |  0.587187 | 0.080151 |
|  2 | + user_product     |  0.592973 | 0.080994 |
|  3 | + all interactions |  0.596135 | 0.081681 |

Adding `user_id` provides the largest jump. `user_product` adds a further **~0.006 ROC-AUC**, and the full interaction bundle reaches **~0.596**. Therefore personalized features improve **ranking quality**; the business should use that lift to improve ad selection, not claim the feature mechanically raises CTR by itself.

## 7. Feature importance and actions

![Feature importance](ctr_importance.png)

Top drivers are `user_id`, `webpage_id`, `campaign_webpage`, `campaign_id`, day-of-week, and product category.

**Actions:** personalize rankers at user level; optimize campaign-placement combinations rather than campaigns in isolation; include webpage context in bid logic; and use time-aware budget pacing.

## 8. SMOTE experiment

|    | variant    |   rows |   positive_rate |   ROC-AUC |   PR-AUC |   Recall |      F1 |   FN |   offline_seconds |
|---:|:-----------|-------:|----------------:|----------:|---------:|---------:|--------:|-----:|------------------:|
|  0 | No SMOTE   |  80000 |        0.070375 |  0.539359 | 0.069267 |  0.41559 | 0.12019 | 2572 |              0.41 |
|  1 | SMOTE 0.30 |  96681 |        0.230769 |  0.536386 | 0.069029 |  0.53783 | 0.12066 | 2034 |              6.31 |
|  2 | SMOTE 1.00 | 148740 |        0.5      |  0.530996 | 0.068437 |  0.52761 | 0.11768 | 2079 |             13.29 |

![SMOTE trade-off](ctr_smote.png)

Partial SMOTE reduces false negatives from **2,572 to 2,034** on the benchmark after threshold tuning, but ROC-AUC and PR-AUC do not improve. Full balance grows the training set by **~1.86x** and slightly reduces F1.

**Decision:** do not use full SMOTE for the production model. Prefer CatBoost/class weighting and threshold optimization. SMOTE increases **offline training cost**, not the number of impressions scored online.

## 9. Inventory application

In [ ]:
# Leakage-safe product demand feature (illustrative)
# hist_product_ctr should be smoothed and computed from prior dates only.
future["expected_clicks"] = future["forecast_impressions"] * future["hist_product_ctr"]
future["expected_orders"] = future["expected_clicks"] * future["click_to_order_rate"]
future["expected_units"] = future["expected_orders"] * future["units_per_order"]
# Add lead time + service level / safety stock for actual replenishment.

CTR is an **upper-funnel demand signal**, not an inventory forecast by itself. Product J has the highest click propensity, but C and H contribute much more click volume. Inventory should therefore use forecast impressions x smoothed CTR x downstream conversion x units/order.

## 10. Final answers to the seven business questions

1. **Weekend vs weekday:** Weekend CTR is 7.33% vs 6.65% weekday (+10.2% relative), but only one weekend day is labeled; validate across more weeks before changing bids.
2. **Products:** J is best by CTR (9.27%); D/H/C follow. G (4.62%) and F (4.91%) are weakest. C leads absolute clicks due to volume.
3. **Personalization:** Yes. ROC-AUC rises from ~0.550 base to ~0.593 after user-product personalization and ~0.596 with all interactions.
4. **Key drivers:** User identity/history, webpage placement, campaign-webpage interaction, campaign, day-of-week, and product category. Amplify them through personalized ranking, placement-aware bidding, and time-aware pacing.
5. **SMOTE:** It reduces false negatives, but does not improve AUC here and materially increases offline training size. Not selected for production.
6. **Product CTR for inventory:** Use smoothed historical CTR as a demand feature; convert expected impressions -> clicks -> orders -> units, then incorporate lead time and safety stock.
7. **User profiles:** Highest robust segments include Male / age level 5 / city index 4 (~8.66%) and Female / age level 5 / city index 4 (~8.47%). Use them as controlled bid modifiers, never as hard exclusions, and require minimum-volume/confidence rules.

## 11. Final production scoring

In [ ]:
# Refit the selected CatBoost model on all labeled data, then score the unlabeled test set.
final_model.fit(X_full_train, train.is_click, cat_features=cat_features)
test_probability = final_model.predict_proba(X_test)[:, 1]
pd.DataFrame({
    "session_id": test.session_id,
    "click_probability": test_probability
}).to_csv("ad_click_test_probabilities.csv", index=False)

Validated full-data scoring produced probabilities with mean ~0.0691, median ~0.0683, 95th percentile ~0.1067, and maximum ~0.1823. The PDF evaluation should rely on the labeled temporal holdout, not on the unlabeled test set.

## 12. Deployment and monitoring recommendations

- Rank ads by probability adjusted for bid/value, not by a fixed 0/1 label alone.
- Tune thresholds to expected business value (`P(click) x value_per_click - cost`), not accuracy.
- Store leakage-safe historical aggregates in an online feature store with freshness windows and smoothing.
- Monitor ROC-AUC, PR-AUC, calibration, CTR, false negatives, feature drift, and segment performance over time.
- Preserve exploration traffic to prevent feedback loops and discover new high-performing ads.
- Treat demographic features carefully for privacy/fairness; behavioral/contextual signals should usually dominate deployment decisions.

**Final selection: CatBoost + personalization/interactions.**